In [6]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

print("Library berhasil di-import!")

Library berhasil di-import!


In [7]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

#menampilkan 5 baris pertama dari data train
train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [8]:
# 1. Reset data gabungan agar aman di-run berkali-kali
test_ids = test_df['PassengerId']
full_df = pd.concat([train_df, test_df], sort=False).reset_index(drop=True)

# 2. Gelar & Keluarga Dasar
full_df['Title'] = full_df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
title_mapping = {'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master'}
full_df['Title'] = full_df['Title'].map(title_mapping).fillna('Rare')
full_df['FamilySize'] = full_df['SibSp'] + full_df['Parch'] + 1

# 3. [FITUR EMAS] Group Survival Rate
# Ambil Nama Belakang (Surname)
full_df['Surname'] = full_df['Name'].apply(lambda x: x.split(',')[0].strip())

# Nilai default 0.5 (Tengah-tengah / Belum diketahui)
full_df['Group_Survival'] = 0.5

# Mengecek riwayat keselamatan berdasarkan Nama Belakang & Harga Tiket
for grp, grp_df in full_df.groupby(['Surname', 'Fare']):
    if len(grp_df) > 1: # Jika ada rombongan
        for ind, row in grp_df.iterrows():
            smax = grp_df.drop(ind)['Survived'].max()
            smin = grp_df.drop(ind)['Survived'].min()
            passID = row['PassengerId']
            if smax == 1.0: # Jika ada anggota rombongan yang selamat
                full_df.loc[full_df['PassengerId'] == passID, 'Group_Survival'] = 1
            elif smin == 0.0: # Jika seluruh rombongan yang tercatat meninggal
                full_df.loc[full_df['PassengerId'] == passID, 'Group_Survival'] = 0

# Mengecek riwayat keselamatan berdasarkan Nomor Tiket (Untuk teman/pengasuh)
for grp, grp_df in full_df.groupby('Ticket'):
    if len(grp_df) > 1:
        for ind, row in grp_df.iterrows():
            if full_df.loc[full_df['PassengerId'] == row['PassengerId'], 'Group_Survival'].values[0] == 0.5:
                smax = grp_df.drop(ind)['Survived'].max()
                smin = grp_df.drop(ind)['Survived'].min()
                passID = row['PassengerId']
                if smax == 1.0:
                    full_df.loc[full_df['PassengerId'] == passID, 'Group_Survival'] = 1
                elif smin == 0.0:
                    full_df.loc[full_df['PassengerId'] == passID, 'Group_Survival'] = 0

print("Cell 3: Fitur Emas 'Group Survival' berhasil dibuat!")

<>:6: SyntaxWarning: invalid escape sequence '\.'
<>:6: SyntaxWarning: invalid escape sequence '\.'
C:\Users\User\AppData\Local\Temp\ipykernel_29416\2249391984.py:6: SyntaxWarning: invalid escape sequence '\.'
  full_df['Title'] = full_df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)


Cell 3: Fitur Emas 'Group Survival' berhasil dibuat!


In [ ]:
# 1. Imputasi standar berdasarkan kelompok
full_df['Age'] = full_df.groupby(['Title', 'Pclass'])['Age'].transform(lambda x: x.fillna(x.median()))
full_df['Fare'] = full_df.groupby('Pclass')['Fare'].transform(lambda x: x.fillna(x.median()))
full_df['Embarked'] = full_df['Embarked'].fillna('S')

# 2. [SOLUSI ERROR] Imputasi "Sapu Jagat" untuk menambal sisa NaN yang terlewat
full_df['Age'] = full_df['Age'].fillna(full_df['Age'].median())
full_df['Fare'] = full_df['Fare'].fillna(full_df['Fare'].median())

# 3. Hapus kolom yang sudah diekstrak maknanya
drop_cols = ['PassengerId', 'Survived', 'Name', 'Ticket', 'Cabin', 'Surname']

# 4. One-Hot Encoding
categorical_cols = ['Sex', 'Embarked', 'Title']
cols_to_dummy = [col for col in categorical_cols if col in full_df.columns]
if cols_to_dummy:
    full_df = pd.get_dummies(full_df, columns=cols_to_dummy, drop_first=True)

# 5. Konversi boolean ke integer
cols_to_convert = [c for c in full_df.columns if full_df[c].dtype == 'bool']
if cols_to_convert:
    full_df[cols_to_convert] = full_df[cols_to_convert].astype(int)

# 6. Pisahkan kembali train dan test
train_clean = full_df[full_df['Survived'].notnull()].copy()
test_clean = full_df[full_df['Survived'].isnull()].copy()

X = train_clean.drop(columns=drop_cols, errors='ignore')
y = train_clean['Survived'].astype(int)
X_test = test_clean.drop(columns=drop_cols, errors='ignore')

# 7. Samakan bentuk matriks
X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

# 8. Pastikan benar-benar tidak ada NaN yang tersisa
X = X.fillna(0)
X_test = X_test.fillna(0)

print(f"Cell 4 & 5 (Fixed): Data Siap! Jumlah Fitur: {X.shape[1]}")

Cell 4 & 5: Data Siap! Jumlah Fitur: 14


In [10]:
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore') # Menyembunyikan peringatan merah dari XGBoost

# 1. Konfigurasi 3 Model Berbeda (Arsitektur Ensemble)
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
xgb = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42, eval_metric='logloss')
lr = LogisticRegression(max_iter=1000, random_state=42)

# 2. Gabungkan ke dalam sistem Voting (Suara terbanyak menang)
ensemble_model = VotingClassifier(estimators=[
    ('Random Forest', rf),
    ('XGBoost', xgb),
    ('Logistic Regression', lr)
], voting='hard')

# 3. Evaluasi performa sistem gabungan
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(ensemble_model, X, y, cv=cv, scoring='accuracy')

print(f"Akurasi Ensemble di Data Train: {scores.mean() * 100:.2f}%")

# 4. Latih model final secara utuh
ensemble_model.fit(X, y)
print("Sistem Ensemble berhasil dilatih secara penuh!")

Akurasi Ensemble di Data Train: 84.18%
Sistem Ensemble berhasil dilatih secara penuh!


In [11]:
# Prediksi data uji menggunakan sistem gabungan
predictions = ensemble_model.predict(X_test)

# Susun hasil akhir
submission = pd.DataFrame({
    'PassengerId': test_ids,
    'Survived': predictions
})

# Ekspor ke file CSV V3
submission.to_csv('submission_titanic_v3_ensemble.csv', index=False)

print("File 'submission_titanic_v3_ensemble.csv' siap! Silakan upload ke Kaggle.")
submission.head()

ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values